# Export Genie Space

This notebook exports a **Databricks Genie space** configuration as a JSON file using the REST API:

```
GET /api/2.0/genie/spaces/{space_id}?include_serialized_space=true
```

The exported JSON contains the full space definition — instructions, sample questions, table configurations, join specs, SQL snippets, and benchmarks.

**Use cases:**
* **Backup** — Save a point-in-time snapshot of a Genie space.
* **Promote across workspaces** — Use the exported JSON with the *Create Genie Space* or *Update Genie Space* API to replicate the space in another workspace.
* **Version control** — Commit the JSON to a Git repository to track configuration changes over time.

**Prerequisites:**
* The caller must have access to the target Genie space.
* The output Volume path must already exist in Unity Catalog (the notebook creates sub-directories as needed).

In [0]:
dbutils.widgets.text(
    "space_id",
    "",
    "Genie Space ID (found in space Settings tab or via List Genie Spaces API)",
)
dbutils.widgets.text(
    "output_volume_path",
    "/Volumes/aira_test/aibi_workshop_schema/export",
    "Unity Catalog Volume path to save the exported JSON",
)

In [0]:
space_id = dbutils.widgets.get("space_id").strip()
output_volume_path = dbutils.widgets.get("output_volume_path").strip()

if not space_id:
    raise ValueError(
        "Parameter 'space_id' is required. "
        "Provide the Genie Space ID via the widget above or as a notebook parameter."
    )

print(f"Space ID           : {space_id}")
print(f"Output Volume Path : {output_volume_path}")

## Call Genie Space REST API
Retrieve the space configuration including the `serialized_space` payload.

In [0]:
import requests

# Obtain workspace URL and PAT from the notebook context
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = _ctx.apiUrl().get()
token = _ctx.apiToken().get()

url = f"{host}/api/2.0/genie/spaces/{space_id}"
params = {"include_serialized_space": "true"}
headers = {"Authorization": f"Bearer {token}"}

try:
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
except requests.exceptions.HTTPError as exc:
    print(f"ERROR — HTTP {response.status_code}")
    print(response.text)
    raise
except requests.exceptions.RequestException as exc:
    print(f"ERROR — Request failed: {exc}")
    raise

space_data = response.json()

print(f"Successfully retrieved Genie space.")
print(f"  Title       : {space_data.get('title', 'N/A')}")
print(f"  Description : {space_data.get('description', 'N/A')}")

## Save to Volume
Persist the exported JSON to the specified Unity Catalog Volume path.

In [0]:
import os
import json
from datetime import datetime, timezone

os.makedirs(output_volume_path, exist_ok=True)

filename = f"genie_space_{space_id}.json"
output_file = os.path.join(output_volume_path, filename)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(space_data, f, indent=2, ensure_ascii=False)

file_size = os.path.getsize(output_file)
timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"File saved  : {output_file}")
print(f"File size   : {file_size:,} bytes")
print(f"Exported at : {timestamp}")
print(f"Top-level keys in exported JSON: {list(space_data.keys())}")

## Preview Exported Content
Quick summary of what was exported, plus a peek inside the `serialized_space` configuration.

In [0]:
import json

print("=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)
print(f"  Space ID     : {space_data.get('space_id', 'N/A')}")
print(f"  Title        : {space_data.get('title', 'N/A')}")
print(f"  Description  : {space_data.get('description', 'N/A')}")
print(f"  Warehouse ID : {space_data.get('warehouse_id', 'N/A')}")
print()

serialized_raw = space_data.get("serialized_space")
if serialized_raw:
    serialized = json.loads(serialized_raw)
    print("Serialized space top-level keys:")
    for key in serialized:
        print(f"  - {key}")
else:
    print("WARNING: 'serialized_space' field is missing from the response.")

print()
print(
    "This file can be used with the Create Genie Space API "
    "(POST /api/2.0/genie/spaces) or the Update Genie Space API "
    "(PUT /api/2.0/genie/spaces/{space_id}) to recreate or "
    "update this space in any workspace."
)